# 02 — Retrieval Ablation
Compare BM25, dense FAISS, and hybrid RRF on NFCorpus. Plots NDCG@10 / MRR@10 / MAP@10.

In [ ]:
import os, sys
os.chdir('..')
from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load retriever components

In [ ]:
import pickle
from pathlib import Path
from scholar.retrieval.bm25 import BM25Index
from scholar.retrieval.dense import DenseRetriever
from scholar.retrieval.hybrid import HybridRetriever

bm25_path = Path('./data/bm25.pkl')
faiss_path = Path(os.environ['FAISS_INDEX_PATH'])
ids_path = faiss_path.with_suffix('.index.ids.json')

print('Loading BM25 index...')
bm25 = BM25Index.load(bm25_path)
print(f'  BM25 docs: {bm25.num_docs:,}')

print('Loading dense retriever...')
dense = DenseRetriever()
dense.load(faiss_path, ids_path)
print(f'  FAISS vectors: {dense._index.ntotal:,}')

hybrid = HybridRetriever(bm25=bm25, dense=dense, k=60)
print('Hybrid retriever ready.')

## 2. Run eval harness

In [ ]:
# Run the retrieval eval script (writes docs/benchmarks.md)
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'scholar.retrieval.eval'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

## 3. Plot results from benchmarks.md

In [ ]:
# Parse the markdown table written by eval.py
import re
bench = Path('./docs/benchmarks.md').read_text()

# Extract retrieval table rows
rows = re.findall(r'\|\s*(BM25|Dense|Hybrid)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|', bench)
if rows:
    results_df = pd.DataFrame(rows, columns=['Method', 'NDCG@10', 'MRR@10', 'MAP@10'])
    for col in ['NDCG@10', 'MRR@10', 'MAP@10']:
        results_df[col] = results_df[col].astype(float)
    print(results_df.to_string(index=False))
else:
    print('No results found — run the eval cell above first.')
    results_df = pd.DataFrame({
        'Method': ['BM25', 'Dense', 'Hybrid'],
        'NDCG@10': [0.0, 0.0, 0.0], 'MRR@10': [0.0, 0.0, 0.0], 'MAP@10': [0.0, 0.0, 0.0]
    })

In [ ]:
metrics = ['NDCG@10', 'MRR@10', 'MAP@10']
x = np.arange(len(metrics))
width = 0.25
colors = ['steelblue', 'darkorange', 'green']

fig, ax = plt.subplots(figsize=(9, 5))
for i, (_, row) in enumerate(results_df.iterrows()):
    ax.bar(x + i*width, [row[m] for m in metrics], width, label=row['Method'], color=colors[i])

ax.set_xticks(x + width); ax.set_xticklabels(metrics)
ax.set_ylim(0, 0.6); ax.set_ylabel('Score')
ax.set_title('Retrieval ablation — NFCorpus')
ax.legend()
plt.tight_layout(); plt.show()

## 4. RRF k sensitivity

In [ ]:
# Quick ablation on k in RRF — compare k=10,30,60,120 on a few queries
from scholar.retrieval.hybrid import _reciprocal_rank_fusion

# Dummy ranked lists
bm25_ranked = [(f'p{i}', i) for i in range(200)]
dense_ranked = [(f'p{200-i}', i) for i in range(200)]

k_values = [10, 30, 60, 120, 200]
overlaps = []
for k in k_values:
    fused = _reciprocal_rank_fusion([bm25_ranked, dense_ranked], k=k)
    top10_ids = {pid for pid, _ in fused[:10]}
    bm25_top10 = {pid for pid, _ in bm25_ranked[:10]}
    overlaps.append(len(top10_ids & bm25_top10))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, overlaps, 'o-', color='teal')
ax.set_xlabel('RRF k'); ax.set_ylabel('Overlap with BM25 top-10')
ax.set_title('RRF k sensitivity (synthetic lists)')
plt.tight_layout(); plt.show()